# ProductIQ Phase 2 — Step 2.4: Data Cleaning

This notebook loads the raw AJIO dataset, runs the cleaning component, and inspects the results.

Cleaning fixes safe formatting issues and flags price anomalies. It does not normalize semantics or modify the raw CSV.

In [1]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path

import pandas as pd

from data.ingestion import AjioDataLoader
from data.preprocessing import AjioDataCleaner, PRICE_ANOMALY_COLUMN


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_PATH = PROJECT_ROOT / "resources" / "raw" / "Ajio Fashion Clothing.csv"

loader = AjioDataLoader()
cleaner = AjioDataCleaner()
raw_dataframe = loader.load(DATASET_PATH)

In [2]:
raw_before_cleaning = deepcopy(raw_dataframe)
cleaning_result = cleaner.clean(raw_dataframe)
cleaned_dataframe = cleaning_result.dataframe
report = cleaning_result.report

print(f"Input rows: {len(raw_dataframe)}")
print(f"Output rows: {len(cleaned_dataframe)}")

Input rows: 367172
Output rows: 367172


In [3]:
print("Before cleaning dtypes (price fields):")
print(raw_dataframe[["Discount Price (in Rs.)", "Original Price (in Rs.)"]].dtypes)
print("\nAfter cleaning dtypes (price fields):")
print(cleaned_dataframe[["Discount Price (in Rs.)", "Original Price (in Rs.)"]].dtypes)

Before cleaning dtypes (price fields):
Discount Price (in Rs.)    object
Original Price (in Rs.)    object
dtype: object

After cleaning dtypes (price fields):
Discount Price (in Rs.)    Int64
Original Price (in Rs.)    Int64
dtype: object


In [4]:
print(f"Price anomalies flagged: {int(cleaned_dataframe[PRICE_ANOMALY_COLUMN].sum())}")
print(f"Report price anomalies: {report.price_anomalies_flagged}")

Price anomalies flagged: 4
Report price anomalies: 4


In [5]:
anomalies = cleaned_dataframe[cleaned_dataframe[PRICE_ANOMALY_COLUMN]][[
    "Id_Product",
    "Brand",
    "Discount Price (in Rs.)",
    "Original Price (in Rs.)",
    PRICE_ANOMALY_COLUMN,
]]
anomalies

,Id_Product,Brand,Discount Price (in Rs.),Original Price (in Rs.),price_anomaly
174758,462983590001,sajasajo,15695,15694,True
315092,462983595001,sajasajo,15695,15694,True
323460,462975646001,sajasajo,14891,14890,True
363335,461716717001,sajasajo,17908,17907,True


In [6]:
print(report.summary())

Input rows: 367172
Output rows: 367172
Rows removed: 0
Columns cleaned: Product_URL, Brand, Description, URL_image, Category_by_gender, Color, Discount Price (in Rs.), Original Price (in Rs.), price_anomaly
Values stripped: 17951
Prices converted: 734344
Price anomalies flagged: 4


In [7]:
pd.testing.assert_frame_equal(raw_dataframe, raw_before_cleaning)
print("Raw loaded DataFrame was not mutated by cleaning.")
print(f"Id_Product unchanged in cleaned output: {cleaned_dataframe['Id_Product'].equals(raw_dataframe['Id_Product'])}")

Raw loaded DataFrame was not mutated by cleaning.
Id_Product unchanged in cleaned output: True


In [8]:
raw_mtime_before = DATASET_PATH.stat().st_mtime
raw_sample = pd.read_csv(DATASET_PATH, nrows=1, encoding="cp1252", dtype=str)
print("Raw CSV still readable and unchanged at path:", DATASET_PATH)
print("Sample original price string from raw CSV:", raw_sample.loc[0, "Original Price (in Rs.)"])

Raw CSV still readable and unchanged at path: E:\AI-ML ENGINEERING\PORTFOLIO-PROJECTS\ProductIQ\resources\raw\Ajio Fashion Clothing.csv
Sample original price string from raw CSV: 699
